In [26]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf, broadcast

In [27]:
spark = SparkSession.builder \
    .appName("Chicago Crimes Analysis") \
    .getOrCreate()

In [28]:
# wczytanie
df = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv("chicago_crimes_sample.csv")

print(df.columns)

['id', 'case_number', 'date', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude', 'location']


In [29]:
# czyszczenie
df_clean = df.dropDuplicates()
print("Po usunięciu duplikatów:")
print("Liczba wierszy:", df_clean.count())
df_clean.show(5, truncate=False)

df_clean = df_clean.dropna(subset=[
    "id",
    "date",
    "primary_type",
    "location_description",
    "year"
])
print("Po usunięciu braków danych:")
print("Liczba wierszy:", df_clean.count())
df_clean.show(5, truncate=False)

df_clean = df_clean.withColumn(
    "date_parsed",
    F.to_timestamp("date", "MM/dd/yyyy hh:mm:ss a")
)
print("Po parsowaniu daty:")
df_clean.select("date", "date_parsed").show(5, truncate=False)

df_clean = df_clean.filter(F.col("date_parsed").isNotNull())
print("Po odfiltrowaniu błędnych dat:")
print("Liczba wierszy:", df_clean.count())
df_clean.select("id", "date", "date_parsed").show(5, truncate=False)

df_clean = df_clean.withColumn("hour", F.hour("date_parsed"))
df_clean = df_clean.withColumn("year", F.year("date_parsed"))

print("Po dodaniu kolumn hour i year:")
df_clean.select("id", "date_parsed", "hour", "year", "primary_type").show(10, truncate=False)

Po usunięciu duplikatów:
Liczba wierszy: 84121
+--------+-----------+-------------------+------------------------+----+-----------------+--------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------+
|id      |case_number|date               |block                   |iucr|primary_type     |description   |location_description|arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year|updated_on         |latitude    |longitude    |location|
+--------+-----------+-------------------+------------------------+----+-----------------+--------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------+
|14191034|JK246130   |2026-05-06 20:30:00|0000X W PARKING LOT E ST|1360|CRIMINAL TRESPASS|TO VEHICLE    |AIRPORT PARKING LOT |false |f

In [30]:
# udf
def pora_dnia(hour):
    if hour is None:
        return "nieznana"
    elif 6 <= hour < 12:
        return "rano"
    elif 12 <= hour < 18:
        return "dzien"
    elif 18 <= hour < 22:
        return "wieczor"
    else:
        return "noc"

pora_dnia_udf = udf(pora_dnia, StringType())

df_clean = df_clean.withColumn("pora_dnia", pora_dnia_udf(F.col("hour")))

print("Po dodaniu kolumny pora_dnia:")
df_clean.select("id", "date_parsed", "hour", "pora_dnia", "primary_type").show(10, truncate=False)

# cache
df_clean.cache()
liczba_wierszy = df_clean.count()

print("Dane zapisane w cache.")
print("Liczba wierszy po czyszczeniu:", liczba_wierszy)

Po dodaniu kolumny pora_dnia:
+--------+-------------------+----+---------+-----------------+
|id      |date_parsed        |hour|pora_dnia|primary_type     |
+--------+-------------------+----+---------+-----------------+
|14191034|2026-05-06 20:30:00|20  |wieczor  |CRIMINAL TRESPASS|
|14189661|2026-05-06 17:00:00|17  |dzien    |THEFT            |
|14188597|2026-05-06 10:42:00|10  |rano     |BATTERY          |
|14188528|2026-05-06 09:50:00|9   |rano     |THEFT            |
|14188152|2026-05-05 18:31:00|18  |wieczor  |THEFT            |
|14187972|2026-05-04 20:30:00|20  |wieczor  |THEFT            |
|14191920|2026-05-04 20:00:00|20  |wieczor  |BURGLARY         |
|14187272|2026-05-04 02:45:00|2   |noc      |BATTERY          |
|14186098|2026-05-04 01:53:00|1   |noc      |ASSAULT          |
|14186908|2026-05-03 21:00:00|21  |wieczor  |BATTERY          |
+--------+-------------------+----+---------+-----------------+
only showing top 10 rows
Dane zapisane w cache.
Liczba wierszy po czyszcze

In [31]:
# typy
print("Najczęstsze typy przestępstw:")

top_crimes = df_clean.groupBy("primary_type") \
    .count() \
    .orderBy(F.desc("count"))

top_crimes.show(10, truncate=False)

Najczęstsze typy przestępstw:
+-------------------+-----+
|primary_type       |count|
+-------------------+-----+
|THEFT              |10920|
|BATTERY            |9253 |
|CRIMINAL DAMAGE    |5497 |
|ASSAULT            |4564 |
|MOTOR VEHICLE THEFT|3886 |
|OTHER OFFENSE      |3459 |
|BURGLARY           |3163 |
|DECEPTIVE PRACTICE |2486 |
|NARCOTICS          |1455 |
|CRIMINAL TRESPASS  |1193 |
+-------------------+-----+
only showing top 10 rows


In [32]:
# lokalizacja
print("Najczęstsze przestępstwa według lokalizacji:")

crime_by_location = df_clean.groupBy("location_description", "primary_type") \
    .count() \
    .orderBy(F.desc("count"))

crime_by_location.show(20, truncate=False)

Najczęstsze przestępstwa według lokalizacji:
+--------------------+-------------------+-----+
|location_description|primary_type       |count|
+--------------------+-------------------+-----+
|APARTMENT           |BATTERY            |3019 |
|STREET              |MOTOR VEHICLE THEFT|2957 |
|STREET              |THEFT              |2288 |
|STREET              |CRIMINAL DAMAGE    |2104 |
|STREET              |BURGLARY           |1540 |
|APARTMENT           |THEFT              |1438 |
|STREET              |BATTERY            |1385 |
|RESIDENCE           |BATTERY            |1347 |
|SMALL RETAIL STORE  |THEFT              |1208 |
|APARTMENT           |OTHER OFFENSE      |1194 |
|APARTMENT           |ASSAULT            |1167 |
|APARTMENT           |CRIMINAL DAMAGE    |1071 |
|DEPARTMENT STORE    |THEFT              |976  |
|STREET              |ASSAULT            |924  |
|RESIDENCE           |OTHER OFFENSE      |891  |
|RESIDENCE           |THEFT              |765  |
|SIDEWALK            |BA

In [33]:
# pora dnia
print("Przestępstwa według pory dnia:")

crime_by_time = df_clean.groupBy("pora_dnia", "primary_type") \
    .count() \
    .orderBy("pora_dnia", F.desc("count"))

crime_by_time.show(30, truncate=False)

Przestępstwa według pory dnia:
+---------+---------------------------------+-----+
|pora_dnia|primary_type                     |count|
+---------+---------------------------------+-----+
|dzien    |THEFT                            |4307 |
|dzien    |BATTERY                          |2824 |
|dzien    |ASSAULT                          |1646 |
|dzien    |CRIMINAL DAMAGE                  |1305 |
|dzien    |OTHER OFFENSE                    |1154 |
|dzien    |DECEPTIVE PRACTICE               |1070 |
|dzien    |MOTOR VEHICLE THEFT              |881  |
|dzien    |BURGLARY                         |691  |
|dzien    |NARCOTICS                        |585  |
|dzien    |CRIMINAL TRESPASS                |426  |
|dzien    |WEAPONS VIOLATION                |276  |
|dzien    |ROBBERY                          |240  |
|dzien    |OFFENSE INVOLVING CHILDREN       |155  |
|dzien    |SEX OFFENSE                      |100  |
|dzien    |PUBLIC PEACE VIOLATION           |93   |
|dzien    |INTERFERENCE WITH PUBL

In [34]:
# aresztowania
print("Liczba aresztowań według typu przestępstwa:")

arrests_by_type = df_clean.groupBy("primary_type", "arrest") \
    .count() \
    .orderBy(F.desc("count"))

arrests_by_type.show(20, truncate=False)

Liczba aresztowań według typu przestępstwa:
+--------------------------+------+-----+
|primary_type              |arrest|count|
+--------------------------+------+-----+
|THEFT                     |false |9927 |
|BATTERY                   |false |7585 |
|CRIMINAL DAMAGE           |false |5244 |
|ASSAULT                   |false |3978 |
|MOTOR VEHICLE THEFT       |false |3736 |
|BURGLARY                  |false |3078 |
|OTHER OFFENSE             |false |2930 |
|DECEPTIVE PRACTICE        |false |2426 |
|BATTERY                   |true  |1668 |
|NARCOTICS                 |true  |1354 |
|THEFT                     |true  |993  |
|ROBBERY                   |false |858  |
|CRIMINAL TRESPASS         |false |782  |
|WEAPONS VIOLATION         |true  |754  |
|ASSAULT                   |true  |586  |
|OTHER OFFENSE             |true  |529  |
|CRIMINAL TRESPASS         |true  |411  |
|OFFENSE INVOLVING CHILDREN|false |342  |
|CRIMINAL SEXUAL ASSAULT   |false |327  |
|SEX OFFENSE               |fals

In [35]:
# agregacja + explain
print("Najcięższa agregacja:")

heavy_agg = df_clean.groupBy(
    "year", "location_description", "primary_type", "pora_dnia"
).agg(
    F.count("*").alias("liczba_zdarzen"),
    F.sum(F.when(F.col("arrest") == True, 1).otherwise(0)).alias("liczba_aresztowan")
).orderBy(F.desc("liczba_zdarzen"))

print("Plan zapytania Spark:")
heavy_agg.explain(True)

print("Wynik najcięższej agregacji:")
heavy_agg.show(20, truncate=False)

Najcięższa agregacja:
Plan zapytania Spark:
== Parsed Logical Plan ==
'Sort ['liczba_zdarzen DESC NULLS LAST], true
+- Aggregate [year#7619, location_description#7217, primary_type#7215, pora_dnia#7659], [year#7619, location_description#7217, primary_type#7215, pora_dnia#7659, count(1) AS liczba_zdarzen#11154L, sum(CASE WHEN (arrest#7218 = true) THEN 1 ELSE 0 END) AS liczba_aresztowan#11155L]
   +- Project [id#7210, case_number#7211, date#7212, block#7213, iucr#7214, primary_type#7215, description#7216, location_description#7217, arrest#7218, domestic#7219, beat#7220, district#7221, ward#7222, community_area#7223, fbi_code#7224, x_coordinate#7225, y_coordinate#7226, year#7619, updated_on#7228, latitude#7229, longitude#7230, location#7231, date_parsed#7507, hour#7618, pora_dnia(hour#7618)#7658 AS pora_dnia#7659]
      +- Project [id#7210, case_number#7211, date#7212, block#7213, iucr#7214, primary_type#7215, description#7216, location_description#7217, arrest#7218, domestic#7219, beat#7

In [36]:
# Parquet
df_clean.write.mode("overwrite") \
    .partitionBy("year") \
    .parquet("chicago_crimes_parquet")

print("Dane zapisane do formatu Parquet w folderze: chicago_crimes_parquet")
print("Partycjonowanie wykonane po kolumnie: year")

Dane zapisane do formatu Parquet w folderze: chicago_crimes_parquet
Partycjonowanie wykonane po kolumnie: year
